In [3]:
import os
import gzip
import subprocess
import pandas as pd
import numpy as np
from datetime import datetime

In [4]:
def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

def get_df(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

In [ ]:
# DATASET = 'Beauty'
# DATASET = 'Video_Games'
# DATASET = 'Grocery_and_Gourmet_Food'
# DATASET = 'Toys'
DATASET = 'FourSquare_NYC'
RAW_PATH = os.path.join('./', DATASET)
# DATA_FILE = 'reviews_{}_5.json.gz'.format(DATASET)
# META_FILE = 'meta_{}.json.gz'.format(DATASET)

RANDOM_SEED = 0
NEG_ITEMS = 999

# Load Data

1. Load interaction data and item metadata
2. Filter out unuseful items in metadata
3. Calculate basic statistics

In [6]:
# # download data if not exists

# if not os.path.exists(RAW_PATH):
#     subprocess.call('mkdir ' + RAW_PATH, shell=True)
# if not os.path.exists(os.path.join(RAW_PATH, DATA_FILE)):
#     print('Downloading interaction data into ' + RAW_PATH)
#     subprocess.call(
#         'cd {} && curl -O http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_{}_5.json.gz'
#         .format(RAW_PATH, DATASET), shell=True)
# if not os.path.exists(os.path.join(RAW_PATH, META_FILE)):
#     print('Downloading item metadata into ' + RAW_PATH)
#     subprocess.call(
#         'cd {} && curl -O http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_{}.json.gz'
#         .format(RAW_PATH, DATASET), shell=True)

In [7]:
# data_df = get_df(os.path.join(RAW_PATH, DATA_FILE))
# data_df.head()

data_df = pd.read_csv(r'./FourSquare_NYC/NYC.csv',sep=',',header='infer',usecols=[5, 8, 2])
data_df.head()
dadaffs = 0


In [8]:
# meta_df = get_df(os.path.join(RAW_PATH, META_FILE))
# meta_df.head()

In [9]:
# Only retain items that appear in interaction data

# useful_meta_df = meta_df[meta_df['asin'].isin(data_df['asin'])].reset_index(drop=True)
# all_items = set(useful_meta_df['asin'].values.tolist())

# def related_filter(related_dict):
#     out_dict = dict()
#     if related_dict is not np.nan:
#         for r in related_dict:
#             out_dict[r] = list(all_items & set(related_dict[r]))
#     return out_dict

# useful_meta_df['related'] = useful_meta_df['related'].apply(related_filter)

### Statistics

In [ ]:
# n_users = data_df['reviewerID'].value_counts().size
# n_items = data_df['asin'].value_counts().size
# n_clicks = len(data_df)
# min_time = data_df['unixReviewTime'].min()
# max_time = data_df['unixReviewTime'].max()

n_users = data_df['user_id'].value_counts().size
n_items = data_df['POI_id'].value_counts().size
n_clicks = len(data_df)
min_time = data_df['UTCTimeOffsetEpoch'].min()
max_time = data_df['UTCTimeOffsetEpoch'].max()

fdajsk=0

In [11]:
time_format = '%Y-%m-%d'

print('# Users:', n_users)
print('# Items:', n_items)
print('# Interactions:', n_clicks)
print('Time Span: {}/{}'.format(
    datetime.utcfromtimestamp(min_time).strftime(time_format),
    datetime.utcfromtimestamp(max_time).strftime(time_format))
)

# Users: 1047
# Items: 4937
# Interactions: 80166
Time Span: 2012-04-03/2012-12-21


C:\Users\dc\AppData\Local\Temp\ipykernel_14184\3410854978.py:7: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  datetime.utcfromtimestamp(min_time).strftime(time_format),
C:\Users\dc\AppData\Local\Temp\ipykernel_14184\3410854978.py:8: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  datetime.utcfromtimestamp(max_time).strftime(time_format))


# Build Dataset

### Interaction data

In [12]:
np.random.seed(RANDOM_SEED)

In [ ]:
# out_df = data_df.rename(columns={'asin': 'item_id', 'reviewerID': 'user_id', 'unixReviewTime': 'time'})
out_df = data_df.rename(columns={'UTCTimeOffsetEpoch': 'time', 'user_id': 'user_id', 'POI_id': 'item_id'})
out_df = out_df[['user_id', 'item_id', 'time']]
out_df = out_df.drop_duplicates(['user_id', 'item_id', 'time'])
out_df = out_df.sort_values(by=['time', 'user_id'], kind='mergesort').reset_index(drop=True)
out_df.head()

KeyError: "['item_id', 'time'] not in index"

In [ ]:
# reindex (start from 1)

uids = sorted(out_df['user_id'].unique())
user2id = dict(zip(uids, range(1, len(uids) + 1)))
iids = sorted(out_df['item_id'].unique())
item2id = dict(zip(iids, range(1, len(iids) + 1)))

out_df['user_id'] = out_df['user_id'].apply(lambda x: user2id[x])
out_df['item_id'] = out_df['item_id'].apply(lambda x: item2id[x])
out_df.head()

,user_id,item_id,time
0,2177,3,965779200
1,4161,18,1068249600
2,4698,23,1073433600
3,10146,6,1075593600
4,3915,126,1082073600


In [ ]:
# leave one out spliting

clicked_item_set = dict()
for user_id, seq_df in out_df.groupby('user_id'):
    clicked_item_set[user_id] = set(seq_df['item_id'].values.tolist())
    
def generate_dev_test(data_df):
    result_dfs = []
    n_items = data_df['item_id'].value_counts().size
    for idx in range(2):
        result_df = data_df.groupby('user_id').tail(1).copy()
        data_df = data_df.drop(result_df.index)
        neg_items = np.random.randint(1, n_items + 1, (len(result_df), NEG_ITEMS))
        for i, uid in enumerate(result_df['user_id'].values):
            user_clicked = clicked_item_set[uid]
            for j in range(len(neg_items[i])):
                while neg_items[i][j] in user_clicked:
                    neg_items[i][j] = np.random.randint(1, n_items + 1)
        result_df['neg_items'] = neg_items.tolist()
        result_dfs.append(result_df)
    return result_dfs, data_df

In [ ]:
leave_df = out_df.groupby('user_id').head(1)
data_df = out_df.drop(leave_df.index)

[test_df, dev_df], data_df = generate_dev_test(data_df)
train_df = pd.concat([leave_df, data_df]).sort_index()

len(train_df), len(dev_df), len(test_df)

(121892, 14681, 14681)

In [ ]:
train_df.head()

,user_id,item_id,time
0,2177,3,965779200
1,4161,18,1068249600
2,4698,23,1073433600
3,10146,6,1075593600
4,3915,126,1082073600


In [ ]:
test_df.head()

,user_id,item_id,time,neg_items
104,6185,762,1149206400,"[2733, 3265, 4860, 7892, 4374, 5875, 6745, 346..."
203,3299,1096,1154044800,"[1877, 3864, 3728, 4269, 5240, 1592, 6446, 713..."
369,12801,319,1158278400,"[4880, 8193, 5412, 5730, 3471, 1799, 5124, 134..."
525,10679,511,1163980800,"[4470, 7854, 7304, 401, 5299, 6952, 1899, 7203..."
528,4501,509,1164067200,"[5681, 5851, 2448, 7024, 2583, 8503, 6964, 528..."


In [ ]:
# save results

train_df.to_csv(os.path.join(RAW_PATH, 'train.csv'), sep='\t', index=False)
dev_df.to_csv(os.path.join(RAW_PATH, 'dev.csv'), sep='\t', index=False)
test_df.to_csv(os.path.join(RAW_PATH, 'test.csv'), sep='\t', index=False)

### Item Metadata

In [ ]:
# level-2 category

# l2_cate_lst = list()
# for cate_lst in useful_meta_df['categories']:
#     l2_cate_lst.append(cate_lst[0][2] if len(cate_lst[0]) > 2 else np.nan)
# useful_meta_df['l2_category'] = l2_cate_lst  
# l2_cates = sorted(useful_meta_df['l2_category'].dropna().unique())
# l2_dict = dict(zip(l2_cates, range(1, len(l2_cates) + 1)))
# useful_meta_df['l2_category'] = useful_meta_df['l2_category'].apply(lambda x: l2_dict[x] if x == x else 0)

In [ ]:
# item_meta_data = dict()
# for idx in range(len(useful_meta_df)):
#     info = useful_meta_df.iloc[idx]['related']
#     item_meta_data[idx] = {
#         'item_id': item2id[useful_meta_df.iloc[idx]['asin']],
#         'i_category': useful_meta_df.iloc[idx]['l2_category'],
#         'r_complement': list(map(lambda x: item2id[x], info['also_bought'])) if 'also_bought' in info else [],
#         'r_substitute': list(map(lambda x: item2id[x], info['also_viewed'])) if 'also_viewed' in info else [],
#     }

# item_meta_df = pd.DataFrame.from_dict(item_meta_data, orient='index')
# item_meta_df = item_meta_df[['item_id', 'i_category', 'r_complement', 'r_substitute']]
# item_meta_df.head()

,item_id,i_category,r_complement,r_substitute
0,1,0,"[1647, 6285, 6068, 4001, 435, 287, 6191, 4169,...","[2286, 194, 6191, 6285, 6068, 6426, 4169]"
1,2,0,"[845, 4420, 3843, 7507, 1987, 1020, 1759, 2575...","[5231, 841, 843, 2355, 845, 2192, 8281, 4777, ..."
2,3,21,"[2135, 4879, 7578, 161, 2901, 2202, 141, 2627]","[4555, 5828, 109, 16]"
3,4,0,"[2057, 7525, 7274, 8528, 8456, 5169, 7299, 807...","[4398, 7274, 4851]"
4,5,0,"[364, 363, 356, 351, 362, 372, 696, 7838, 690,...","[364, 363, 355, 356, 351, 361, 352, 357, 362, ..."


In [ ]:
# save results

# item_meta_df.to_csv(os.path.join(RAW_PATH, 'item_meta.csv'), sep='\t', index=False)